# Chapter 4 Student Colab Notebook: Vector Spaces

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 4.1-4.8  
**Audience:** Student version

This notebook is the complete interactive teaching companion for Chapter 4. It combines theory, original worked examples, detailed row-reduction and coordinate calculations, geometric interpretations, applications, Python visualizations, interactive demonstrations, section worksheets, and complete solutions.

> **How to use this student notebook**
>
> Run the cells from top to bottom in Google Colab. Read the full theory and worked examples, explore the visualizations, and complete each section worksheet. Complete instructor solutions are not included. Display mathematics uses `$$...$$` for reliable Colab rendering.

## Chapter map

1. **Section 4.1:** Vectors in $\mathbb{R}^n$  
2. **Section 4.2:** Vector Spaces  
3. **Section 4.3:** Subspaces of Vector Spaces  
4. **Section 4.4:** Spanning Sets and Linear Independence  
5. **Section 4.5:** Basis and Dimension  
6. **Section 4.6:** Rank of a Matrix and Systems of Linear Equations  
7. **Section 4.7:** Coordinates and Change of Basis  
8. **Section 4.8:** Applications of Vector Spaces  

Each section ends with a terminology table, a worksheet, and complete instructor solutions.

In [ ]:
# Core imports used throughout Chapter 4
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

sp.init_printing()

def show_matrix(M, label=None):
    M = sp.Matrix(M)
    if label:
        display(Math(rf"{label}={sp.latex(M)}"))
    else:
        display(M)
    return M

def show_vector(v, label=None):
    v = sp.Matrix(v)
    if label:
        display(Math(rf"{label}={sp.latex(v)}"))
    else:
        display(v)
    return v

def show_rref(A, label='A'):
    A = sp.Matrix(A)
    R, pivots = A.rref()
    display(Math(rf"{label}={sp.latex(A)}"))
    display(Math(rf"\operatorname{{rref}}({label})={sp.latex(R)}"))
    print('Pivot-column indices (0-based):', pivots)
    return R, pivots

def plot_vector2(v, label='v', origin=(0,0), ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,5))
    ox, oy = origin
    vx, vy = float(v[0]), float(v[1])
    ax.quiver(ox, oy, vx, vy, angles='xy', scale_units='xy', scale=1)
    ax.text(ox+vx, oy+vy, f'  {label}')
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.grid(alpha=0.3)
    return ax

def row_reduce_with_steps(M, operations, labels=None):
    """Apply explicit row operations to a SymPy matrix and display every step."""
    M = sp.Matrix(M)
    display(Math(rf"M_0={sp.latex(M)}"))
    for k, op in enumerate(operations, start=1):
        op(M)
        text = labels[k-1] if labels else f'Step {k}'
        display(Markdown(f'**{text}**'))
        display(Math(rf"M_{k}={sp.latex(M)}"))
    return M

def coordinate_vector(B, x):
    Bm = sp.Matrix.hstack(*[sp.Matrix(v) for v in B])
    return Bm.LUsolve(sp.Matrix(x))

> **Original-example policy**
>
> All worked examples and numerical values are original. Personal-interest themes are used selectively, including the San Antonio Spurs, hiking, badminton, driving, travel, Backstreet Boys, and R&B music. Any prices, routes, coordinates, or datasets are fictional teaching models unless explicitly identified otherwise.

# Section 4.1 - Vectors in $\mathbb{R}^n$

## Learning objectives

Students should be able to:

- represent vectors geometrically as directed line segments;
- add, subtract, and scale vectors in $\mathbb{R}^2$, $\mathbb{R}^3$, and $\mathbb{R}^n$;
- interpret vector operations geometrically;
- use vector algebra properties;
- write one vector as a linear combination of other vectors.

### Definition: vector in the plane

> A vector $\mathbf{v}=(v_1,v_2)$ can be represented by a directed segment from the origin to $(v_1,v_2)$. The numbers $v_1$ and $v_2$ are its components.

## Original application: hiking displacement

A hiker travels according to two displacement vectors, measured in kilometers:

$$
\mathbf{u}=(4,3), \qquad \mathbf{v}=(-1,5).
$$

The total displacement is

$$
\mathbf{u}+\mathbf{v}=(4-1,3+5)=(3,8).
$$

The vector $-\mathbf{u}=(-4,-3)$ reverses the first displacement, while $\frac12\mathbf{u}=(2,\frac32)$ has the same direction and half the magnitude.

In [ ]:
u = np.array([4.,3.])
v = np.array([-1.,5.])
fig, ax = plt.subplots(figsize=(7,6))
plot_vector2(u, 'u', ax=ax)
plot_vector2(v, 'v', ax=ax)
plot_vector2(u+v, 'u+v', ax=ax)
ax.set_xlim(-2,6); ax.set_ylim(-1,9); ax.set_aspect('equal')
ax.set_title('Hiking displacement vectors')
plt.show()

## The parallelogram rule

The vector sum can be visualized as the diagonal of a parallelogram. Equivalently, translate the second vector so that its initial point is the terminal point of the first.

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
plot_vector2(u,'u',ax=ax)
plot_vector2(v,'v',origin=u,ax=ax)
plot_vector2(u+v,'u+v',ax=ax)
ax.plot([0,v[0],u[0]+v[0],u[0],0],[0,v[1],u[1]+v[1],u[1],0],linestyle='--')
ax.set_xlim(-2,6); ax.set_ylim(-1,9); ax.set_aspect('equal'); ax.set_title('Parallelogram rule')
plt.show()

> **Vector operations in $\mathbb{R}^n$**
>
> For $\mathbf{u}=(u_1,\ldots,u_n)$, $\mathbf{v}=(v_1,\ldots,v_n)$, and scalar $c$, define $\mathbf{u}+\mathbf{v}=(u_1+v_1,\ldots,u_n+v_n)$ and $c\mathbf{u}=(cu_1,\ldots,cu_n)$.

## Spurs-inspired vector arithmetic in $\mathbb{R}^3$

Let

$$
\mathbf{a}=(9,1,20), \qquad \mathbf{b}=(21,-3,4).
$$

Then

$$
\mathbf{a}+\mathbf{b}=(30,-2,24),
$$

$$
2\mathbf{a}=(18,2,40),
$$

and

$$
\mathbf{b}-2\mathbf{a}=(3,-5,-36).
$$

The entries are Spurs-inspired teaching values; they do not represent a single real dataset.

In [ ]:
a = sp.Matrix([9,1,20])
b = sp.Matrix([21,-3,4])
show_vector(a+b, r'\mathbf{a}+\mathbf{b}')
show_vector(2*a, r'2\mathbf{a}')
show_vector(b-2*a, r'\mathbf{b}-2\mathbf{a}')

## Solving a vector equation in $\mathbb{R}^4$

Let

$$
\mathbf{u}=(2,-1,3,0),\quad
\mathbf{v}=(1,4,-2,5),\quad
\mathbf{w}=(-1,2,1,-3).
$$

Solve

$$
3(\mathbf{x}+\mathbf{w})=2\mathbf{u}-\mathbf{v}+\mathbf{x}.
$$

Move the $\mathbf{x}$ terms together:

$$
3\mathbf{x}+3\mathbf{w}=2\mathbf{u}-\mathbf{v}+\mathbf{x},
$$

$$
2\mathbf{x}=2\mathbf{u}-\mathbf{v}-3\mathbf{w},
$$

so

$$
\mathbf{x}=\frac12\left(2\mathbf{u}-\mathbf{v}-3\mathbf{w}\right).
$$

Substitution gives

$$
\mathbf{x}=\left(3,-6,\frac52,2\right).
$$

An intermediate algebra step can make the computation easier to follow:

$$
\frac12(2\mathbf u-\mathbf v-3\mathbf w)
=
\mathbf u-\frac12\mathbf v-\frac32\mathbf w.
$$


In [ ]:
u4=sp.Matrix([2,-1,3,0]); v4=sp.Matrix([1,4,-2,5]); w4=sp.Matrix([-1,2,1,-3])
x4=sp.Rational(1,2)*(2*u4-v4-3*w4)
show_vector(x4, r'\mathbf{x}')
print('Verification:', 3*(x4+w4) == 2*u4-v4+x4)

### Definition: linear combination

> A vector $\mathbf{x}$ is a linear combination of $\mathbf{v}_1,\ldots,\mathbf{v}_k$ if $\mathbf{x}=c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k$ for scalars $c_1,\ldots,c_k$.

## Detailed linear-combination example

Let

$$
\mathbf{v}_1=(1,2,0),\quad
\mathbf{v}_2=(0,1,1),\quad
\mathbf{v}_3=(2,0,1),
$$

and let $\mathbf{x}=(7,4,5)$. We seek $c_1,c_2,c_3$ such that

$$
\mathbf{x}=c_1\mathbf{v}_1+c_2\mathbf{v}_2+c_3\mathbf{v}_3.
$$

Equating components gives

$$
\begin{aligned}
c_1+2c_3&=7,\\
2c_1+c_2&=4,\\
c_2+c_3&=5.
\end{aligned}
$$

The augmented matrix is

$$
\left[
\begin{array}{ccc|c}
1&0&2&7\\
2&1&0&4\\
0&1&1&5
\end{array}
\right].
$$

Apply $R_2\leftarrow R_2-2R_1$:

$$
\left[
\begin{array}{ccc|c}
1&0&2&7\\
0&1&-4&-10\\
0&1&1&5
\end{array}
\right].
$$

Then $R_3\leftarrow R_3-R_2$:

$$
\left[
\begin{array}{ccc|c}
1&0&2&7\\
0&1&-4&-10\\
0&0&5&15
\end{array}
\right].
$$

Thus $c_3=3$, $c_2=2$, and $c_1=1$. Therefore,

$$
\boxed{\mathbf{x}=\mathbf{v}_1+2\mathbf{v}_2+3\mathbf{v}_3}.
$$

In [ ]:
V = sp.Matrix.hstack(sp.Matrix([1,2,0]), sp.Matrix([0,1,1]), sp.Matrix([2,0,1]))
x = sp.Matrix([7,4,5])
show_matrix(V, 'V')
show_vector(V.LUsolve(x), r'\mathbf{c}')

> **Common mistakes**
>
> Do not confuse a vector with its length. Add vectors componentwise, not by adding lengths. A linear combination problem is a system for unknown coefficients.

> **Closing note**
>
> Vector operations are the computational language of the rest of the chapter. Spanning, independence, basis, coordinates, and rank all ask increasingly structural questions about linear combinations.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Vector | Ordered list of components; geometrically, a directed displacement. |
| Component | One coordinate of a vector. |
| Zero vector | Vector whose components are all zero. |
| Vector addition | Componentwise addition. |
| Scalar multiplication | Multiplication of every component by the same scalar. |
| Additive inverse | $-\mathbf{v}$, satisfying $\mathbf{v}+(-\mathbf{v})=\mathbf{0}$. |
| Linear combination | Sum $c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k$. |

## Section 4.1 worksheet

1. Let $\mathbf{u}=(5,-2)$ and $\mathbf{v}=(-3,7)$. Find $\mathbf{u}+\mathbf{v}$, $2\mathbf{u}-\mathbf{v}$, and $-\frac12\mathbf{v}$.
2. Let $\mathbf{a}=(2,1,-1)$, $\mathbf{b}=(0,3,2)$, and $\mathbf{c}=(1,-2,4)$. Compute $3\mathbf{a}-2\mathbf{b}+\mathbf{c}$.
3. Solve $2(\mathbf{x}-\mathbf{u})=\mathbf{v}+\mathbf{x}$ for $\mathbf{x}$ when $\mathbf{u}=(1,2,3)$ and $\mathbf{v}=(4,-1,2)$.
4. Determine coefficients $c_1,c_2$ such that $(9,21)=c_1(1,1)+c_2(1,-1)$.
5. A badminton player makes court-movement vectors $(2,1)$, $(-1,3)$, and $(0,-2)$ meters. Find the total displacement and interpret it.

# Section 4.2 - Vector Spaces

## Learning objectives

Students should be able to:

- state and interpret the ten vector-space axioms;
- recognize standard vector spaces such as $\mathbb{R}^n$, $P_n$, $M_{m,n}$, and spaces of functions;
- verify selected axioms for an unfamiliar set and operations;
- disprove vector-space status efficiently by finding one failed axiom.

### Definition: vector space

> A vector space is a set $V$ equipped with vector addition and scalar multiplication satisfying closure, associativity, commutativity of addition, a zero vector, additive inverses, two distributive laws, compatibility of scalar multiplication, and scalar identity.

## The ten axioms

For all $\mathbf{u},\mathbf{v},\mathbf{w}\in V$ and scalars $c,d$:

1. $\mathbf{u}+\mathbf{v}\in V$.
2. $\mathbf{u}+\mathbf{v}=\mathbf{v}+\mathbf{u}$.
3. $(\mathbf{u}+\mathbf{v})+\mathbf{w}=\mathbf{u}+(\mathbf{v}+\mathbf{w})$.
4. There is $\mathbf{0}\in V$ such that $\mathbf{u}+\mathbf{0}=\mathbf{u}$.
5. For every $\mathbf{u}$ there is $-\mathbf{u}$ with $\mathbf{u}+(-\mathbf{u})=\mathbf{0}$.
6. $c\mathbf{u}\in V$.
7. $c(\mathbf{u}+\mathbf{v})=c\mathbf{u}+c\mathbf{v}$.
8. $(c+d)\mathbf{u}=c\mathbf{u}+d\mathbf{u}$.
9. $c(d\mathbf{u})=(cd)\mathbf{u}$.
10. $1\mathbf{u}=\mathbf{u}$.

## Important standard vector spaces

- $\mathbb{R}^n$: all real $n$-tuples;
- $P_n$: all polynomials of degree at most $n$, together with the zero polynomial;
- $M_{m,n}$: all $m\times n$ real matrices;
- $C[a,b]$: continuous real-valued functions on $[a,b]$;
- solution sets of homogeneous linear systems.

## Example: $M_{2,3}$ is a vector space

If $A$ and $B$ are $2\times3$ matrices, then $A+B$ is also $2\times3$. If $c$ is a scalar, then $cA$ is also $2\times3$. Thus closure holds. The remaining axioms follow from ordinary real-number arithmetic applied entrywise. The zero vector is the zero matrix and the additive inverse of $A$ is $-A$.

### What is being checked?

We are verifying the vector-space axioms, not merely listing matrix operations. In particular, addition and scalar multiplication remain in $M_{2,3}$, the zero matrix belongs to the space, and every $A$ has an additive inverse $-A$ in the same space.


In [ ]:
A = sp.Matrix([[1,9,21],[20,4,-3]])
B = sp.Matrix([[2,-1,5],[0,7,1]])
show_matrix(A+B, 'A+B')
show_matrix(-A, '-A')

## Example: $P_2$ is a vector space

A polynomial in $P_2$ has the form

$$
p(x)=a_0+a_1x+a_2x^2.
$$

The sum of two such polynomials and any scalar multiple remain of degree at most two. The zero polynomial belongs to $P_2$, and $-p(x)$ is also in $P_2$. Therefore $P_2$ is a vector space under ordinary polynomial operations.

> **Fast disproof strategy**
>
> To show that a set is not a vector space, one counterexample is sufficient. Common targets are failure to contain the zero vector, failure of closure under addition, or failure of closure under multiplication by a negative scalar.

## Non-example: polynomials of exactly degree two

Let $W$ be the set of all polynomials of exactly degree two. Consider

$$
p(x)=x^2+x, \qquad q(x)=-x^2+2.
$$

Both are in $W$, but

$$
p(x)+q(x)=x+2,
$$

which has degree one. Thus $W$ is not closed under addition and is not a vector space.

## Non-example: a positivity restriction

Let

$$
W=\{(x,y)\in\mathbb{R}^2:x>0\}.
$$

The vector $(1,0)$ lies in $W$, but

$$
-1(1,0)=(-1,0)\notin W.
$$

So $W$ is not closed under scalar multiplication. It also does not contain the zero vector.

## Optional example: unusual operations

Let $V=\mathbb{R}$ with operations

$$
u\oplus v=u+v-1,
$$

and

$$
c\odot u=cu-c+1.
$$

The zero vector for $\oplus$ is $1$, not $0$, because $u\oplus1=u$. This example illustrates that the axioms concern the specified operations, not necessarily the usual ones.

In [ ]:
def add_star(u,v): return u+v-1
def scale_star(c,u): return c*u-c+1
for u0 in [0,2,5]:
    print(u0, add_star(u0,1), scale_star(1,u0), scale_star(0,u0))

> **Closing note**
>
> A vector space is an algebraic environment in which linear combinations behave predictably. The elements need not be arrows; they may be matrices, polynomials, functions, signals, or data objects.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Vector space | Set with addition and scalar multiplication satisfying ten axioms. |
| Closure | An operation keeps results inside the set. |
| Additive identity | The zero vector of the space. |
| Additive inverse | Vector that sums with a given vector to the zero vector. |
| $P_n$ | Polynomials of degree at most $n$. |
| $M_{m,n}$ | Real $m\times n$ matrices. |
| $C[a,b]$ | Continuous functions on $[a,b]$. |

## Section 4.2 worksheet

For each set with the usual operations, determine whether it is a vector space. Justify your answer.

1. All vectors $(x,y,z)$ satisfying $x+y-z=0$.
2. All vectors $(x,y)$ satisfying $x+y=1$.
3. All polynomials of degree at most three.
4. All $2\times2$ matrices with determinant $1$.
5. All continuous functions $f$ satisfying $f(0)=0$.
6. Explain why testing only several numerical examples cannot prove all ten axioms.

# Section 4.3 - Subspaces of Vector Spaces

## Learning objectives

Students should be able to:

- apply the subspace test;
- use the zero-vector test to reject affine sets quickly;
- recognize lines and planes through the origin as subspaces;
- analyze subspaces of $\mathbb{R}^n$, polynomial spaces, and matrix spaces;
- understand intersections of subspaces.

### Definition and subspace test

Let $W\subseteq V$. Then $W$ is a **subspace of the parent space $V$** when:

1. the zero vector of $V$ is in $W$;
2. $\mathbf u,\mathbf v\in W$ implies $\mathbf u+\mathbf v\in W$;
3. $c\mathbf u\in W$ for every scalar $c$.

All three conditions must hold.


### Definition: subspace

> A nonempty subset $W$ of a vector space $V$ is a subspace when it is itself a vector space using the same addition and scalar multiplication as $V$.

### Subspace test

> A nonempty subset $W\subseteq V$ is a subspace if and only if it is closed under addition and scalar multiplication. Equivalently, for all $\mathbf{u},\mathbf{v}\in W$ and scalars $a,b$, the combination $a\mathbf{u}+b\mathbf{v}$ lies in $W$.

## Original example: a plane through the origin

Let

$$
W=\{(x,y,z)\in\mathbb{R}^3:x-2y+z=0\}.
$$

The zero vector belongs to $W$. If $\mathbf{u},\mathbf{v}\in W$, then

$$
u_1-2u_2+u_3=0,
\qquad
v_1-2v_2+v_3=0.
$$

For scalars $a,b$,

$$
(a\mathbf{u}+b\mathbf{v})_1-2(a\mathbf{u}+b\mathbf{v})_2+(a\mathbf{u}+b\mathbf{v})_3
=a(0)+b(0)=0.
$$

Hence $W$ is a subspace.

Solving for $x$ gives

$$
x=2y-z.
$$

Let $y=s$ and $z=t$. Then

$$
(x,y,z)=s(2,1,0)+t(-1,0,1).
$$

In [ ]:
# Plot part of the plane x - 2y + z = 0, or z = -x + 2y.
xx, yy = np.meshgrid(np.linspace(-3,3,25), np.linspace(-3,3,25))
zz = -xx + 2*yy
fig = plt.figure(figsize=(7,6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(xx,yy,zz,alpha=0.5)
ax.scatter([0],[0],[0],s=50)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Subspace plane through the origin')
plt.show()

## Non-example: a parallel affine plane

Consider

$$
U=\{(x,y,z):x-2y+z=1\}.
$$

The zero vector does not satisfy the equation. Therefore $U$ is not a subspace. Geometrically, it is parallel to $W$ but shifted away from the origin.

## Singular matrices are not a subspace

Let $S$ be the set of singular $2\times2$ matrices. The matrices

$$
A=\begin{bmatrix}1&0\\0&0\end{bmatrix},
\qquad
B=\begin{bmatrix}0&0\\0&1\end{bmatrix}
$$

are singular, but

$$
A+B=I_2
$$

is invertible. Thus $S$ is not closed under addition.

In [ ]:
A_sing=sp.Matrix([[1,0],[0,0]])
B_sing=sp.Matrix([[0,0],[0,1]])
print(A_sing.det(), B_sing.det(), (A_sing+B_sing).det())

## Intersection of two subspaces

Let

$$
W_1=\{(x,y,z):x+y+z=0\},
$$

and

$$
W_2=\{(x,y,z):x-z=0\}.
$$

Their intersection satisfies both equations. Since $x=z$, the first equation gives $2x+y=0$, so $y=-2x$. Therefore,

$$
W_1\cap W_2=\{t(1,-2,1):t\in\mathbb{R}\},
$$

which is a line through the origin and hence a subspace.

## A polynomial subspace

The set

$$
W=\{p\in P_3:p(1)=0\}
$$

is a subspace. If $p(1)=q(1)=0$, then

$$
(p+q)(1)=p(1)+q(1)=0,
$$

and

$$
(cp)(1)=c\,p(1)=0.
$$

Every polynomial in $W$ is divisible by $x-1$.

> **Common mistake**
>
> Containing the zero vector is necessary but not sufficient. After the zero-vector check, closure still must be established.

> **Closing note**
>
> Subspaces are the linear subsets of a vector space. Homogeneous equations naturally produce subspaces because zero right-hand sides are preserved by linear combinations.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Subspace | Subset that is a vector space under inherited operations. |
| Subspace test | Check nonemptiness, closure under addition, and closure under scalar multiplication. |
| Affine set | Shifted linear set; generally not a subspace unless it passes through the origin. |
| Intersection | Vectors belonging to both subspaces. |
| Homogeneous constraint | Linear condition with zero right-hand side. |

## Section 4.3 worksheet

Determine whether each set is a subspace of the stated vector space.

1. $W=\{(x,y,z):2x-y+3z=0\}\subseteq\mathbb{R}^3$.
2. $W=\{(x,y,z):2x-y+3z=7\}\subseteq\mathbb{R}^3$.
3. $W=\{p\in P_3:p(0)=p(1)=0\}$.
4. The set of symmetric $2\times2$ matrices.
5. The set of invertible $2\times2$ matrices.
6. Find a parametric description and spanning set for $\{(x,y,z):x+y-z=0\}$.

# Section 4.4 - Spanning Sets and Linear Independence

## Learning objectives

Students should be able to:

- determine whether a vector is a linear combination of a set;
- describe and visualize spans;
- determine whether a set spans a vector space;
- test linear independence by solving a homogeneous system;
- identify explicit dependence relations.

### Definition: span

> The span of $S=\{\mathbf{v}_1,\ldots,\mathbf{v}_k\}$ is the set of all linear combinations $c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k$. It is the smallest subspace containing $S$.

## Linear-combination example revisited

For

$$
\mathbf{v}_1=(1,2,0),\quad
\mathbf{v}_2=(0,1,1),\quad
\mathbf{v}_3=(2,0,1),
$$

we found

$$
(7,4,5)=\mathbf{v}_1+2\mathbf{v}_2+3\mathbf{v}_3.
$$

Therefore $(7,4,5)$ belongs to $\operatorname{span}\{\mathbf{v}_1,\mathbf{v}_2,\mathbf{v}_3\}$.

## Does the set span $\mathbb{R}^3$?

Place the vectors as columns:

$$
A=\begin{bmatrix}
1&0&2\\
2&1&0\\
0&1&1
\end{bmatrix}.
$$

Because

$$
\det(A)=5\ne0,
$$

the system $A\mathbf{c}=\mathbf{x}$ has a unique solution for every $\mathbf{x}\in\mathbb{R}^3$. Hence the three vectors span $\mathbb{R}^3$.

To span $\mathbb R^3$ means every target $\mathbf b\in\mathbb R^3$ can be written as a linear combination of the given vectors. Equivalently, $A\mathbf c=\mathbf b$ must be solvable for every $\mathbf b$.


In [ ]:
A_span=sp.Matrix([[1,0,2],[2,1,0],[0,1,1]])
print('det(A)=',A_span.det(),'rank(A)=',A_span.rank())

### Definition: linear independence

> The set $\{\mathbf{v}_1,\ldots,\mathbf{v}_k\}$ is linearly independent if $c_1\mathbf{v}_1+\cdots+c_k\mathbf{v}_k=\mathbf{0}$ has only the trivial solution. Otherwise the set is linearly dependent.

## Original dependent set in $\mathbb{R}^3$

Let

$$
\mathbf{a}=(1,2,3),\quad
\mathbf{b}=(2,1,0),\quad
\mathbf{c}=(5,4,3).
$$

Observe that

$$
\mathbf{c}=\mathbf{a}+2\mathbf{b}.
$$

Therefore,

$$
\mathbf{a}+2\mathbf{b}-\mathbf{c}=\mathbf{0}
$$

is a nontrivial dependence relation. The set is linearly dependent.

In [ ]:
D=sp.Matrix.hstack(sp.Matrix([1,2,3]),sp.Matrix([2,1,0]),sp.Matrix([5,4,3]))
show_matrix(D,'D')
print('rank=',D.rank(),'nullspace=',D.nullspace())

## Detailed independence test

Consider

$$
S=\{(1,1,0),(0,1,1),(1,0,1)\}.
$$

Set

$$
c_1(1,1,0)+c_2(0,1,1)+c_3(1,0,1)=\mathbf{0}.
$$

This gives

$$
\begin{aligned}
c_1+c_3&=0,\\
c_1+c_2&=0,\\
c_2+c_3&=0.
\end{aligned}
$$

The augmented matrix is

$$
\left[
\begin{array}{ccc|c}
1&0&1&0\\
1&1&0&0\\
0&1&1&0
\end{array}
\right].
$$

After $R_2\leftarrow R_2-R_1$,

$$
\left[
\begin{array}{ccc|c}
1&0&1&0\\
0&1&-1&0\\
0&1&1&0
\end{array}
\right].
$$

After $R_3\leftarrow R_3-R_2$,

$$
\left[
\begin{array}{ccc|c}
1&0&1&0\\
0&1&-1&0\\
0&0&2&0
\end{array}
\right].
$$

Thus $c_3=0$, then $c_2=0$, and $c_1=0$. The set is linearly independent.

## Polynomial dependence

In $P_2$, let

$$
p_1(x)=1+x,\qquad p_2(x)=x+x^2,\qquad p_3(x)=1+2x+x^2.
$$

Since

$$
p_3=p_1+p_2,
$$

we have $p_1+p_2-p_3=0$. The set is linearly dependent.

## Geometry of spans in $\mathbb{R}^3$

- One nonzero vector spans a line through the origin.
- Two independent vectors span a plane through the origin.
- Three independent vectors span all of $\mathbb{R}^3$.
- Adding a vector already in the span does not enlarge the span.

In [ ]:
# Visualize the plane spanned by two independent vectors.
a=np.array([2.,1.,0.]); b=np.array([-1.,1.,2.])
s,t=np.meshgrid(np.linspace(-2,2,15),np.linspace(-2,2,15))
pts=s[...,None]*a+t[...,None]*b
fig=plt.figure(figsize=(7,6)); ax=fig.add_subplot(111,projection='3d')
ax.plot_surface(pts[:,:,0],pts[:,:,1],pts[:,:,2],alpha=0.45)
ax.quiver(0,0,0,*a); ax.quiver(0,0,0,*b)
ax.set_title('Span of two independent vectors'); plt.show()

> **Closing note**
>
> Span asks whether vectors provide enough directions; independence asks whether any direction is redundant. A basis will require both.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Span | Set of all linear combinations of a set of vectors. |
| Spanning set | Set whose span equals the whole target space. |
| Linearly independent | Only the trivial homogeneous coefficient solution exists. |
| Linearly dependent | A nontrivial coefficient relation produces zero. |
| Trivial solution | All coefficients equal zero. |
| Dependence relation | Explicit nontrivial linear combination equal to zero. |

## Section 4.4 worksheet

1. Determine whether $(8,1,7)$ is a linear combination of $(1,2,0)$, $(0,1,1)$, and $(2,0,1)$.
2. Determine whether $\{(1,0,1),(0,1,1),(1,1,2)\}$ is linearly independent.
3. Does $\{(1,2),(3,6)\}$ span $\mathbb{R}^2$? Explain.
4. Determine whether $\{1+x,\ x+x^2,\ 1+2x+x^2\}$ is independent in $P_2$.
5. Find a nontrivial dependence relation among $(1,0,2)$, $(0,1,1)$, and $(2,1,5)$.
6. Explain geometrically why two nonparallel vectors in $\mathbb{R}^2$ span the plane.

# Section 4.5 - Basis and Dimension

## Learning objectives

Students should be able to:

- test whether a set is a basis;
- use standard bases for $\mathbb{R}^n$, $P_n$, and $M_{m,n}$;
- explain uniqueness of basis representation;
- find a basis for a subspace;
- compute dimensions and use dimension-based basis tests.

### Basis as spanning without redundancy

For $P_2$, the standard set $\{1,x,x^2\}$ spans because every polynomial $a+bx+cx^2$ is a linear combination of these three functions. It is independent because

$$
c_1+c_2x+c_3x^2=0
$$

forces $c_1=c_2=c_3=0$.

In an $n$-dimensional space: $n$ independent vectors form a basis; $n$ spanning vectors form a basis; more than $n$ vectors are dependent; fewer than $n$ cannot span the whole space.


### Definition: basis

> A basis for $V$ is a set that spans $V$ and is linearly independent. A basis is therefore complete without redundancy.

## Standard bases

The standard basis for $\mathbb{R}^3$ is

$$
\mathbf{e}_1=(1,0,0),\quad \mathbf{e}_2=(0,1,0),\quad \mathbf{e}_3=(0,0,1).
$$

The standard basis for $P_2$ is

$$
\{1,x,x^2\}.
$$

A standard basis for $M_{2,2}$ is

$$
E_{11}=\begin{bmatrix}1&0\\0&0\end{bmatrix},\quad
E_{12}=\begin{bmatrix}0&1\\0&0\end{bmatrix},\quad
E_{21}=\begin{bmatrix}0&0\\1&0\end{bmatrix},\quad
E_{22}=\begin{bmatrix}0&0\\0&1\end{bmatrix}.
$$

## Original basis example in $\mathbb{R}^3$

Let

$$
B=\{(1,2,0),(0,1,1),(2,0,1)\}.
$$

The basis matrix is

$$
M_B=\begin{bmatrix}1&0&2\\2&1&0\\0&1&1\end{bmatrix}.
$$

Because $\det(M_B)=5\ne0$, the columns are independent and span $\mathbb{R}^3$. Thus $B$ is a basis.

### Uniqueness theorem

> If $B$ is a basis for $V$, every vector in $V$ has exactly one representation as a linear combination of the vectors in $B$.

## Extracting a basis from a spanning set

Consider

$$
S=\{\mathbf{v}_1,\mathbf{v}_2,\mathbf{v}_3,\mathbf{v}_4\},
$$

where

$$
\mathbf{v}_1=(1,2,0),\quad
\mathbf{v}_2=(0,1,1),\quad
\mathbf{v}_3=(2,0,1),\quad
\mathbf{v}_4=(1,3,1).
$$

Since $\mathbf{v}_4=\mathbf{v}_1+\mathbf{v}_2$, it is redundant. The first three vectors form a basis for the same span.

In [ ]:
Smat=sp.Matrix.hstack(sp.Matrix([1,2,0]),sp.Matrix([0,1,1]),sp.Matrix([2,0,1]),sp.Matrix([1,3,1]))
R,pivs=Smat.rref()
show_matrix(Smat,'S')
show_matrix(R,r'\operatorname{rref}(S)')
print('Pivot columns:',pivs)

### Definition: dimension

> If every basis of $V$ contains $n$ vectors, then $\dim(V)=n$. The zero vector space has dimension zero.

## Dimension of common spaces

$$
\dim(\mathbb{R}^n)=n,
\qquad
\dim(P_n)=n+1,
\qquad
\dim(M_{m,n})=mn.
$$

## Dimension of a parametrized subspace

Let

$$
W=\{(a+b,a-b,2a):a,b\in\mathbb{R}\}.
$$

Rewrite a general vector:

$$
(a+b,a-b,2a)=a(1,1,2)+b(1,-1,0).
$$

The two spanning vectors are not scalar multiples, so they are independent. Therefore

$$
\dim(W)=2.
$$

## Polynomial subspace with a condition

Let

$$
W=\{p\in P_3:p(1)=0\}.
$$

Every $p\in W$ is divisible by $x-1$. One basis is

$$
\{x-1,\ x(x-1),\ x^2(x-1)\}
=
\{x-1,\ x^2-x,\ x^3-x^2\}.
$$

Thus $\dim(W)=3$.

## Basis tests in an $n$-dimensional space

If $\dim(V)=n$, then:

- any independent set of exactly $n$ vectors is a basis;
- any spanning set of exactly $n$ vectors is a basis;
- any set with more than $n$ vectors is dependent;
- any set with fewer than $n$ vectors cannot span $V$.

> **Closing note**
>
> Dimension measures the number of independent directions in a space. A basis provides a minimal coordinate system for those directions.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Basis | Linearly independent spanning set. |
| Standard basis | Natural coordinate basis for a familiar vector space. |
| Finite-dimensional | Has a basis with finitely many vectors. |
| Dimension | Number of vectors in any basis. |
| Basis extraction | Removing redundant vectors while preserving span. |
| Unique representation | Each vector has exactly one coefficient list relative to a basis. |

## Section 4.5 worksheet

1. Determine whether $\{(1,0,1),(0,1,1),(1,1,0)\}$ is a basis for $\mathbb{R}^3$.
2. Find the dimension of $\operatorname{span}\{(1,2,0),(0,1,1),(1,3,1)\}$.
3. Give a standard basis and dimension for $P_4$.
4. Give a basis and dimension for the symmetric $2\times2$ matrices.
5. Find a basis and dimension for $W=\{(s+t,s-t,3s):s,t\in\mathbb{R}\}$.
6. Explain why five vectors in $P_3$ must be linearly dependent.

# Section 4.6 - Rank of a Matrix and Systems of Linear Equations

## Learning objectives

Students should be able to:

- find bases for row and column spaces;
- compute rank and nullity;
- find a basis for the nullspace;
- apply the rank-nullity theorem;
- express nonhomogeneous solutions as $\mathbf{x}_p+\mathbf{x}_h$;
- interpret consistency using the column space.

For a nonhomogeneous system, the goal is all solutions:

$$
\boxed{\mathbf x=\mathbf x_p+\mathbf x_h},
$$

where $\mathbf x_p$ is one particular solution and $\mathbf x_h\in\operatorname{Null}(A)$.


### Rank and rank-nullity

$$
\operatorname{rank}(A)=\dim(\operatorname{Col}(A))
$$

is the number of pivot columns. If $A$ has $n$ columns,

$$
\boxed{\operatorname{rank}(A)+\operatorname{nullity}(A)=n}.
$$

This balances directions that contribute to the range with directions lost in the nullspace.


### Definitions: row and column spaces

> For an $m\times n$ matrix $A$, the row space is the span of the row vectors in $\mathbb{R}^n$. The column space is the span of the column vectors in $\mathbb{R}^m$.

## A shared matrix for the section

Let

$$
A=\begin{bmatrix}
1&1&1&0&1\\
2&1&3&1&3\\
1&0&2&2&5
\end{bmatrix}.
$$

Its reduced row-echelon form is

$$
R=\begin{bmatrix}
1&0&2&0&-1\\
0&1&-1&0&2\\
0&0&0&1&3
\end{bmatrix}.
$$

The pivot columns are $1$, $2$, and $4$.

In [ ]:
A46=sp.Matrix([[1,1,1,0,1],[2,1,3,1,3],[1,0,2,2,5]])
R46,p46=A46.rref()
show_matrix(A46,'A')
show_matrix(R46,'R')
print('Pivots:',p46,'rank:',A46.rank())

## Basis for the row space

The nonzero rows of the row-echelon form form a basis for the row space:

$$
\left\{
(1,0,2,0,-1),
(0,1,-1,0,2),
(0,0,0,1,3)
\right\}.
$$

## Basis for the column space

Use the pivot positions from $R$, but select the corresponding columns from the original matrix $A$:

$$
\left\{
\begin{bmatrix}1\\2\\1\end{bmatrix},
\begin{bmatrix}1\\1\\0\end{bmatrix},
\begin{bmatrix}0\\1\\2\end{bmatrix}
\right\}.
$$

Do not use the columns of the RREF as a basis for the original column space.

### Definition: rank

> The rank of a matrix is the common dimension of its row space and column space. Here $\operatorname{rank}(A)=3$.

### Definition: nullspace

> The nullspace $N(A)$ is the set of all solutions of $A\mathbf{x}=\mathbf{0}$. It is a subspace of $\mathbb{R}^n$. Its dimension is the nullity.

## Detailed nullspace calculation

From the RREF,

$$
\begin{aligned}
x_1+2x_3-x_5&=0,\\
x_2-x_3+2x_5&=0,\\
x_4+3x_5&=0.
\end{aligned}
$$

Let the free variables be

$$
x_3=s,\qquad x_5=t.
$$

Then

$$
x_1=-2s+t,
\qquad
x_2=s-2t,
\qquad
x_4=-3t.
$$

Therefore,

$$
\mathbf{x}=
\begin{bmatrix}-2s+t\\s-2t\\s\\-3t\\t\end{bmatrix}
=s\begin{bmatrix}-2\\1\\1\\0\\0\end{bmatrix}
+t\begin{bmatrix}1\\-2\\0\\-3\\1\end{bmatrix}.
$$

A basis for $N(A)$ is

$$
\left\{
\begin{bmatrix}-2\\1\\1\\0\\0\end{bmatrix},
\begin{bmatrix}1\\-2\\0\\-3\\1\end{bmatrix}
\right\}.
$$

In [ ]:
null_basis=A46.nullspace()
for i,v in enumerate(null_basis,1): show_vector(v,rf'\mathbf{{n}}_{i}')
print('rank + nullity =',A46.rank()+len(null_basis),'= number of columns')

### Rank-nullity theorem

> For an $m\times n$ matrix, $n=\operatorname{rank}(A)+\operatorname{nullity}(A)$. Here $5=3+2$.

## Nonhomogeneous systems: $\mathbf{x}_p+\mathbf{x}_h$

Consider

$$
A\mathbf{x}=\mathbf{b},
\qquad
\mathbf{b}=\begin{bmatrix}6\\12\\14\end{bmatrix}.
$$

One particular solution is

$$
\mathbf{x}_p=\begin{bmatrix}1\\2\\0\\-1\\3\end{bmatrix},
$$

because $A\mathbf{x}_p=\mathbf{b}$. Every solution is

$$
\mathbf{x}=\mathbf{x}_p+\mathbf{x}_h,
$$

where $\mathbf{x}_h\in N(A)$. Hence

$$
\mathbf{x}=
\begin{bmatrix}1\\2\\0\\-1\\3\end{bmatrix}
+s\begin{bmatrix}-2\\1\\1\\0\\0\end{bmatrix}
+t\begin{bmatrix}1\\-2\\0\\-3\\1\end{bmatrix}.
$$

In [ ]:
xp=sp.Matrix([1,2,0,-1,3]); b46=A46*xp
show_vector(b46,r'\mathbf{b}')
print('Verification A*xp=b:',A46*xp==b46)

## Consistency and the column space

The system $A\mathbf{x}=\mathbf{b}$ is consistent if and only if $\mathbf{b}$ lies in $\operatorname{Col}(A)$. The coefficients of a solution express $\mathbf{b}$ as a linear combination of the columns of $A$.

## Equivalent conditions for a square matrix

For an $n\times n$ matrix $A$, the following are equivalent:

- $A$ is invertible;
- $A\mathbf{x}=\mathbf{b}$ has a unique solution for every $\mathbf{b}$;
- $A\mathbf{x}=\mathbf{0}$ has only the trivial solution;
- $A$ is row-equivalent to $I_n$;
- $\det(A)\ne0$;
- $\operatorname{rank}(A)=n$;
- the rows are independent;
- the columns are independent;
- the columns span $\mathbb{R}^n$.

> **Common mistake**
>
> Pivot columns are located using the RREF, but a column-space basis must be taken from the original matrix.

> **Closing note**
>
> Rank counts independent output directions; nullity counts independent input directions that are lost. Together they account for every input coordinate.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Row space | Span of matrix rows. |
| Column space | Span of matrix columns. |
| Rank | Dimension of row or column space. |
| Nullspace | Solution space of $A\mathbf{x}=0$. |
| Nullity | Dimension of the nullspace. |
| Pivot column | Column corresponding to a leading entry in echelon form. |
| Particular solution | One solution of a nonhomogeneous system. |
| Homogeneous solution | Any vector in the nullspace. |
| Rank-nullity theorem | $n=\operatorname{rank}(A)+\operatorname{nullity}(A)$. |

## Section 4.6 worksheet

Let

$$
B=\begin{bmatrix}
1&2&0&1\\
2&4&1&3\\
-1&-2&1&0
\end{bmatrix}.
$$

1. Find the RREF and rank of $B$.
2. Find a basis for the row space.
3. Find a basis for the column space using columns of the original matrix.
4. Find a basis for $N(B)$ and the nullity.
5. Verify rank-nullity.
6. If $\mathbf{x}_p=(1,0,2,-1)^T$, compute $\mathbf{b}=B\mathbf{x}_p$ and write the complete solution of $B\mathbf{x}=\mathbf{b}$ as $\mathbf{x}_p+\mathbf{x}_h$.

# Section 4.7 - Coordinates and Change of Basis

## Learning objectives

Students should be able to:

- find coordinates relative to an ordered basis;
- reconstruct a vector from its coordinates;
- compute transition matrices between bases;
- convert coordinate vectors in both directions;
- represent polynomials and matrices using nonstandard bases.

### Ordered bases and the goal of transition matrices

Coordinates require an **ordered basis**

$$
B=(\mathbf b_1,\ldots,\mathbf b_n).
$$

The transition matrix has the explicit purpose

$$
[\mathbf v]_C=P_{C\leftarrow B}[\mathbf v]_B.
$$


### Definition: coordinate vector

> If $B=\{\mathbf{v}_1,\ldots,\mathbf{v}_n\}$ is an ordered basis and $\mathbf{x}=c_1\mathbf{v}_1+\cdots+c_n\mathbf{v}_n$, then $[\mathbf{x}]_B=(c_1,\ldots,c_n)^T$.

## Coordinates in a nonstandard basis

Let

$$
B=\{(1,1),(1,-1)\},
\qquad
\mathbf{x}=(9,21).
$$

We solve

$$
c_1(1,1)+c_2(1,-1)=(9,21).
$$

This gives

$$
c_1+c_2=9,
\qquad
c_1-c_2=21.
$$

Adding the equations gives $2c_1=30$, so $c_1=15$. Then $c_2=-6$. Therefore,

$$
[\mathbf{x}]_B=\begin{bmatrix}15\\-6\end{bmatrix}.
$$

In [ ]:
B2=[sp.Matrix([1,1]),sp.Matrix([1,-1])]
x2=sp.Matrix([9,21])
show_vector(coordinate_vector(B2,x2),r'[\mathbf{x}]_B')

## Coordinate geometry

The vector itself does not change when the basis changes. Only its coordinate description changes. Relative to the standard basis,

$$
[\mathbf{x}]_S=\begin{bmatrix}9\\21\end{bmatrix},
$$

while relative to $B$,

$$
[\mathbf{x}]_B=\begin{bmatrix}15\\-6\end{bmatrix}.
$$

In [ ]:
fig,ax=plt.subplots(figsize=(7,6))
plot_vector2([1,1],'b1',ax=ax)
plot_vector2([1,-1],'b2',ax=ax)
plot_vector2([9,21],'x',ax=ax)
ax.set_xlim(-3,11); ax.set_ylim(-3,23); ax.set_aspect('equal'); ax.set_title('One vector, two coordinate descriptions')
plt.show()

### Definition: transition matrix

> The transition matrix $P_{C\leftarrow B}$ satisfies $[\mathbf{x}]_C=P_{C\leftarrow B}[\mathbf{x}]_B$. Its columns are the $C$-coordinates of the vectors in $B$.

## Detailed $3\times3$ change of basis

Let

$$
B=\{(1,1,0),(0,1,1),(1,0,1)\},
$$

and

$$
C=\{(1,0,0),(1,1,0),(1,1,1)\}.
$$

The basis matrices are

$$
M_B=\begin{bmatrix}1&0&1\\1&1&0\\0&1&1\end{bmatrix},
\qquad
M_C=\begin{bmatrix}1&1&1\\0&1&1\\0&0&1\end{bmatrix}.
$$

Since

$$
\mathbf{x}=M_B[\mathbf{x}]_B=M_C[\mathbf{x}]_C,
$$

we obtain

$$
[\mathbf{x}]_C=M_C^{-1}M_B[\mathbf{x}]_B.
$$

Thus

$$
P_{C\leftarrow B}=M_C^{-1}M_B
=\begin{bmatrix}
0&-1&1\\
1&0&-1\\
0&1&1
\end{bmatrix}.
$$

In [ ]:
MB=sp.Matrix([[1,0,1],[1,1,0],[0,1,1]])
MC=sp.Matrix([[1,1,1],[0,1,1],[0,0,1]])
P_CB=MC.inv()*MB
show_matrix(MB,'M_B'); show_matrix(MC,'M_C'); show_matrix(P_CB,r'P_{C\leftarrow B}')
show_matrix(P_CB.inv(),r'P_{B\leftarrow C}')

## Gauss-Jordan construction of the transition matrix

To compute $P_{C\leftarrow B}$ without separately finding $M_C^{-1}$, row reduce

$$
[M_C\mid M_B].
$$

Start with

$$
\left[
\begin{array}{ccc|ccc}
1&1&1&1&0&1\\
0&1&1&1&1&0\\
0&0&1&0&1&1
\end{array}
\right].
$$

Apply $R_2\leftarrow R_2-R_3$:

$$
\left[
\begin{array}{ccc|ccc}
1&1&1&1&0&1\\
0&1&0&1&0&-1\\
0&0&1&0&1&1
\end{array}
\right].
$$

Apply $R_1\leftarrow R_1-R_3$:

$$
\left[
\begin{array}{ccc|ccc}
1&1&0&1&-1&0\\
0&1&0&1&0&-1\\
0&0&1&0&1&1
\end{array}
\right].
$$

Apply $R_1\leftarrow R_1-R_2$:

$$
\left[
\begin{array}{ccc|ccc}
1&0&0&0&-1&1\\
0&1&0&1&0&-1\\
0&0&1&0&1&1
\end{array}
\right].
$$

The right block is $P_{C\leftarrow B}$.

## Converting a vector

For

$$
\mathbf{x}=(9,20,21),
$$

one finds

$$
[\mathbf{x}]_B=\begin{bmatrix}4\\16\\5\end{bmatrix}.
$$

Then

$$
[\mathbf{x}]_C=P_{C\leftarrow B}[\mathbf{x}]_B
=\begin{bmatrix}-11\\-1\\21\end{bmatrix}.
$$

In [ ]:
x3=sp.Matrix([9,20,21])
cB=MB.LUsolve(x3); cC=MC.LUsolve(x3)
show_vector(cB,r'[\mathbf{x}]_B'); show_vector(P_CB*cB,r'P_{C\leftarrow B}[\mathbf{x}]_B'); show_vector(cC,r'[\mathbf{x}]_C')

## Coordinates in $P_3$

Relative to the ordered basis

$$
B=\{1,\ x-1,\ (x-1)^2,\ (x-1)^3\},
$$

write

$$
p(x)=2+3x-x^2+x^3.
$$

Expanding around $x=1$ gives

$$
p(x)=5+4(x-1)+2(x-1)^2+(x-1)^3.
$$

Therefore,

$$
[p]_B=\begin{bmatrix}5\\4\\2\\1\end{bmatrix}.
$$

In [ ]:
x=sp.symbols('x')
p=2+3*x-x**2+x**3
q=sp.expand(p.subs(x,sp.symbols('t')+1))
print(q)

> **Common mistake**
>
> A transition matrix depends on direction. Always write the arrow notation $P_{C\leftarrow B}$ before calculating.

> **Closing note**
>
> Coordinates are labels relative to a basis. Change of basis changes the labels, not the underlying vector.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Ordered basis | Basis whose vector order is fixed. |
| Coordinate vector | Coefficients of a vector relative to an ordered basis. |
| Basis matrix | Matrix whose columns are basis vectors. |
| Transition matrix | Matrix converting coordinates from one basis to another. |
| Change of basis | Re-expression of the same vector in a different coordinate system. |

## Section 4.7 worksheet

1. For $B=\{(1,2),(2,-1)\}$, find $[(8,7)]_B$.
2. Reconstruct $\mathbf{x}$ from $[\mathbf{x}]_B=(3,-2)^T$ using the same basis.
3. Let $B=\{(1,1),(1,-1)\}$ and $S$ be the standard basis. Find $P_{S\leftarrow B}$ and $P_{B\leftarrow S}$.
4. Use the transition matrix to convert $[\mathbf{x}]_B=(5,4)^T$ to standard coordinates.
5. Find the coordinates of $p(x)=3-2x+4x^2$ relative to $\{1,x,x^2\}$.
6. Find the coordinates of the same polynomial relative to $\{1,x-1,(x-1)^2\}$.

# Section 4.8 - Applications of Vector Spaces

## Learning objectives

Students should be able to:

- verify solutions of linear homogeneous differential equations;
- compute and interpret Wronskians;
- test linear independence of solution functions;
- identify standard conic sections;
- eliminate an $xy$ term by rotation of axes;
- visualize rotated conics computationally.

The vector-space connection is independence: a nonzero Wronskian shows that the solution functions are linearly independent and can serve as a basis for the solution space.

The Chapter 4 connection is **change of basis**: a suitable coordinate change reveals the natural axes of a quadratic form. Later, eigenvectors will give a systematic way to find those directions.


## Linear homogeneous differential equations

A linear differential equation of order $n$ has the form

$$
y^{(n)}+g_{n-1}(x)y^{(n-1)}+\cdots+g_1(x)y'+g_0(x)y=f(x).
$$

It is homogeneous when $f(x)=0$. The solution set of a linear homogeneous differential equation is a vector space because sums and scalar multiples of solutions remain solutions.

## Original example: verifying two solutions

Consider

$$
y''-y=0.
$$

For $y_1=e^x$,

$$
y_1''-y_1=e^x-e^x=0.
$$

For $y_2=e^{-x}$,

$$
y_2''-y_2=e^{-x}-e^{-x}=0.
$$

Thus both functions are solutions.

### Definition: Wronskian

> For functions $y_1,\ldots,y_n$, the Wronskian is the determinant whose rows contain the functions and their derivatives through order $n-1$. For solutions of an $n$th-order homogeneous linear equation, a Wronskian not identically zero is equivalent to linear independence.

## Wronskian of $e^x$ and $e^{-x}$

$$
W(y_1,y_2)=
\det\begin{bmatrix}
e^x&e^{-x}\\
e^x&-e^{-x}
\end{bmatrix}
=-1-1=-2.
$$

Since $W=-2\ne0$, the functions are linearly independent. Therefore the general solution is

$$
y=C_1e^x+C_2e^{-x}.
$$

In [ ]:
x=sp.symbols('x')
y1=sp.exp(x); y2=sp.exp(-x)
W2=sp.wronskian([y1,y2],x)
print('Wronskian:',sp.simplify(W2))

## Third-order example

The functions $1$, $x$, and $x^2$ solve $y^{(3)}=0$. Their Wronskian is

$$
W=
\det\begin{bmatrix}
1&x&x^2\\
0&1&2x\\
0&0&2
\end{bmatrix}=2.
$$

They are linearly independent, so every solution is

$$
y=C_1+C_2x+C_3x^2.
$$

In [ ]:
print(sp.wronskian([1,x,x**2],x))

## Standard conic forms

Common conics include:

- circle: $(x-h)^2+(y-k)^2=r^2$;
- ellipse: $\frac{(x-h)^2}{a^2}+\frac{(y-k)^2}{b^2}=1$;
- hyperbola: $\frac{(x-h)^2}{a^2}-\frac{(y-k)^2}{b^2}=1$;
- parabola: $(x-h)^2=4p(y-k)$ or $(y-k)^2=4p(x-h)$.

## Original conic examples

A fictional hiking-path parabola is

$$
(x-2)^2=-8(y-1).
$$

Its vertex is $(2,1)$, $4p=-8$, so $p=-2$, and the parabola opens downward.

A fictional travel-region ellipse is

$$
\frac{(x+1)^2}{9}+\frac{(y-2)^2}{4}=1.
$$

Its center is $(-1,2)$, with semiaxes $3$ and $2$.

In [ ]:
t=np.linspace(0,2*np.pi,400)
xe=-1+3*np.cos(t); ye=2+2*np.sin(t)
yp=1-(np.linspace(-4,4,300)**2)/8
xp=2+np.linspace(-4,4,300)
fig,ax=plt.subplots(figsize=(7,6))
ax.plot(xe,ye,label='Ellipse')
ax.plot(xp,yp,label='Parabola')
ax.axhline(0,linewidth=.8); ax.axvline(0,linewidth=.8); ax.grid(alpha=.3); ax.axis('equal'); ax.legend(); ax.set_title('Two standard conics')
plt.show()

## Rotation of axes

Consider

$$
5x^2-6xy+5y^2=20.
$$

The quadratic-form matrix is

$$
Q=\begin{bmatrix}5&-3\\-3&5\end{bmatrix}.
$$

Its orthonormal eigenvectors point along the rotated axes, with eigenvalues $2$ and $8$. Using the $45^\circ$ rotation

$$
x=\frac{x'-y'}{\sqrt2},
\qquad
y=\frac{x'+y'}{\sqrt2},
$$

substitution yields

$$
2x'^2+8y'^2=20.
$$

Therefore,

$$
\frac{x'^2}{10}+\frac{y'^2}{5/2}=1,
$$

which is an ellipse whose principal axes are rotated relative to the original $x$- and $y$-axes.

In [ ]:
Q=sp.Matrix([[5,-3],[-3,5]])
print('Eigenstructure:',Q.eigenvects())
# Plot the rotated ellipse directly from the principal-axis parametrization.
t=np.linspace(0,2*np.pi,500)
xp=np.sqrt(10)*np.cos(t); yp=np.sqrt(2.5)*np.sin(t)
R=np.array([[1/np.sqrt(2),-1/np.sqrt(2)],[1/np.sqrt(2),1/np.sqrt(2)]])
xy=R@np.vstack([xp,yp])
fig,ax=plt.subplots(figsize=(7,6))
ax.plot(xy[0],xy[1],label='5x² - 6xy + 5y² = 20')
ax.axhline(0,linewidth=.8); ax.axvline(0,linewidth=.8); ax.grid(alpha=.3); ax.axis('equal'); ax.legend(); ax.set_title('Rotated ellipse')
plt.show()

In [ ]:
# Interactive rotation explorer. This cell remains useful even when widgets are unavailable.
def rotated_ellipse(theta_degrees=45):
    theta=np.deg2rad(theta_degrees)
    t=np.linspace(0,2*np.pi,400)
    base=np.vstack([3*np.cos(t),1.5*np.sin(t)])
    R=np.array([[np.cos(theta),-np.sin(theta)],[np.sin(theta),np.cos(theta)]])
    pts=R@base
    plt.figure(figsize=(6,6)); plt.plot(pts[0],pts[1]); plt.axhline(0,linewidth=.8); plt.axvline(0,linewidth=.8); plt.grid(alpha=.3); plt.axis('equal'); plt.title(f'Rotated ellipse: {theta_degrees} degrees'); plt.show()

if WIDGETS_AVAILABLE:
    interact(rotated_ellipse, theta_degrees=IntSlider(min=0,max=90,step=5,value=45));
else:
    rotated_ellipse(45)

> **Closing note**
>
> Wronskians translate functional independence into a determinant test. Rotation of axes translates a quadratic equation into coordinates aligned with its natural geometry.

## Definitions and terminology

| Term | Meaning |
|---|---|
| Linear differential equation | Differential equation linear in $y$ and its derivatives. |
| Homogeneous | Right-hand side equals zero. |
| Wronskian | Determinant built from functions and derivatives. |
| Conic section | Circle, ellipse, parabola, or hyperbola. |
| Rotation of axes | Orthogonal coordinate change used to remove an $xy$ term. |
| Principal axes | Natural axes aligned with a conic's quadratic form. |

## Section 4.8 worksheet

1. Verify that $\cos x$ and $\sin x$ solve $y''+y=0$.
2. Compute their Wronskian and determine independence.
3. Compute the Wronskian of $1$, $x$, and $x^2$.
4. Identify the center and semiaxes of $\frac{(x-3)^2}{16}+\frac{(y+1)^2}{9}=1$.
5. Identify the vertex and opening direction of $(y-2)^2=-12(x+1)$.
6. Rotate $4x^2+4xy+4y^2=12$ by $45^\circ$ and classify the resulting conic.